# 01 — Pipeline de datos: Extracción, Carga y Análisis SQL
Este notebook ejecuta el pipeline completo del proyecto para las 17 series del Banco Central de Chile:
1. **Extracción y Carga:** Petición a la API SieteRestWS y persistencia idempotente en SQLite.
2. **Exploración Relacional:** Verificación del esquema `temas`, `series` y `observaciones`.
3. **Análisis Macro con SQL:** JOINs, agregaciones mensuales y variación interanual con funciones de ventana (`LAG`).

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

# Permitir importaciones desde la carpeta src/
sys.path.append(str(Path.cwd().parent / "src"))

import config
import carga_db

# Fijar MODO DEMO en False si ya cuentas con tu .env configurado
DEMO = False
FECHA_INICIO = "2010-01-01"
FECHA_CORTE = "2026-08-01"

# 1. Ejecución del pipeline de extracción y carga
print(f"Iniciando pipeline a SQLite ({FECHA_INICIO} a {FECHA_CORTE})...")
carga_db.main(demo=DEMO, desde=FECHA_INICIO, hasta=FECHA_CORTE)

Iniciando pipeline a SQLite (2010-01-01 a 2026-08-01)...
[OK] Esquema SQL verificado / creado.
[OK] Catálogo cargado: 17 series del BCCh.

Iniciando pipeline | Modo: API Banco Central de Chile | Desde 2010-01-01 hasta 2026-08-01

  [OK] Imacec empalmado: 200 observaciones, hasta 2026-08-01
  [OK] Imacec minero: 200 observaciones, hasta 2026-08-01
  [OK] Imacec industria: 200 observaciones, hasta 2026-08-01
  [OK] Imacec resto bienes: 200 observaciones, hasta 2026-08-01
  [OK] Imacec comercio: 200 observaciones, hasta 2026-08-01
  [OK] Imacec servicios: 200 observaciones, hasta 2026-08-01
  [OK] Imacec impuestos: 164 observaciones, hasta 2026-08-01
  [OK] Precio cobre: 200 observaciones, hasta 2026-08-01
  [OK] Tipo cambio observado: 4129 observaciones, hasta 2026-07-31
  [OK] Exportaciones cobre: 200 observaciones, hasta 2026-08-01
  [OK] Exportaciones litio: 200 observaciones, hasta 2026-08-01
  [OK] Exportaciones agropecuarias: 200 observaciones, hasta 2026-08-01
  [OK] Importaciones

In [2]:
# Conexión SQLAlchemy a la base de datos coyuntura.db
engine = carga_db.obtener_engine()

# Verificar las 3 tablas relacionales del proyecto
with engine.connect() as conn:
    tablas = pd.read_sql(text("SELECT name FROM sqlite_master WHERE type IN ('table','view')"), conn)

tablas

,name
0,temas
1,series
2,observaciones


In [3]:
# Catálogo relacional cruzando las 18 series con sus 4 temas oficiales
consulta_join = text("""
SELECT 
    s.id_serie, 
    s.nombre, 
    t.tema, 
    s.frecuencia, 
    s.unidad
FROM series s
JOIN temas t ON t.id_tema = s.id_tema
ORDER BY t.tema, s.nombre;
""")

with engine.connect() as conn:
    catalogo = pd.read_sql(consulta_join, conn)

catalogo

,id_serie,nombre,tema,frecuencia,unidad
0,F032.IMC.IND.Z.Z.EP18.COM.Z.0.M,Imacec comercio,Actividad,M,Indice 2018=100
1,F032.IMC.IND.Z.Z.EP18.Z.Z.0.M,Imacec empalmado,Actividad,M,Indice 2018=100
2,F032.IMC.IND.Z.Z.EP18.IMP.Z.0.M,Imacec impuestos,Actividad,M,Indice 2018=100
3,F032.IMC.IND.Z.Z.EP18.04.Z.0.M,Imacec industria,Actividad,M,Indice 2018=100
4,F032.IMC.IND.Z.Z.EP18.03.Z.0.M,Imacec minero,Actividad,M,Indice 2018=100
5,F032.IMC.IND.Z.Z.EP18.RB.Z.0.M,Imacec resto bienes,Actividad,M,Indice 2018=100
6,F032.IMC.IND.Z.Z.EP18.SERV.Z.0.M,Imacec servicios,Actividad,M,Indice 2018=100
7,F089.IPC.V12.14.M,Expectativa inflacion 11 meses,Expectativas,M,Porcentaje
8,F089.TCN.PRE.14.M,Expectativa tipo cambio 11 meses,Expectativas,M,CLP/USD
9,F049.DES.TAS.INE9.10.M,Tasa desocupacion,Mercado Laboral,M,Porcentaje


In [4]:
# Resumen de cobertura temporal y recuento de observaciones por serie
consulta_groupby = text("""
SELECT 
    t.tema,
    s.nombre AS serie,
    MIN(o.fecha) AS primera_obs,
    MAX(o.fecha) AS ultima_obs,
    COUNT(*) AS total_observaciones
FROM observaciones o
JOIN series s ON s.id_serie = o.id_serie
JOIN temas t ON t.id_tema = s.id_tema
GROUP BY t.tema, s.nombre
ORDER BY t.tema, s.nombre;
""")

with engine.connect() as conn:
    cobertura = pd.read_sql(consulta_groupby, conn)

cobertura

,tema,serie,primera_obs,ultima_obs,total_observaciones
0,Actividad,Imacec comercio,2010-01-01,2026-08-01,200
1,Actividad,Imacec empalmado,2010-01-01,2026-08-01,200
2,Actividad,Imacec impuestos,2013-01-01,2026-08-01,164
3,Actividad,Imacec industria,2010-01-01,2026-08-01,200
4,Actividad,Imacec minero,2010-01-01,2026-08-01,200
5,Actividad,Imacec resto bienes,2010-01-01,2026-08-01,200
6,Actividad,Imacec servicios,2010-01-01,2026-08-01,200
7,Expectativas,Expectativa inflacion 11 meses,2010-01-01,2026-08-01,200
8,Expectativas,Expectativa tipo cambio 11 meses,2010-01-01,2026-08-01,200
9,Mercado Laboral,Tasa desocupacion,2010-03-01,2026-08-01,198


In [5]:
# Variación interanual (% 12 meses) procesando datos diarios a nivel mensual
consulta_ventana = text("""
WITH mensual AS (
    SELECT 
        o.id_serie,
        strftime('%Y-%m', o.fecha) AS mes,
        AVG(o.valor) AS valor_promedio
    FROM observaciones AS o
    GROUP BY o.id_serie, mes
),
variaciones AS (
    SELECT 
        m.id_serie,
        m.mes,
        m.valor_promedio,
        LAG(m.valor_promedio, 12) OVER (
            PARTITION BY m.id_serie 
            ORDER BY m.mes
        ) AS valor_hace_12m,
        100.0 * (m.valor_promedio / LAG(m.valor_promedio, 12) OVER (
            PARTITION BY m.id_serie 
            ORDER BY m.mes
        ) - 1.0) AS var_interanual_pct
    FROM mensual AS m
)
SELECT 
    t.tema,
    s.nombre AS serie,
    v.mes,
    ROUND(v.valor_promedio, 2) AS valor_mensual,
    ROUND(v.var_interanual_pct, 2) AS var_12m_pct
FROM variaciones AS v
JOIN series AS s ON s.id_serie = v.id_serie
JOIN temas AS t ON t.id_tema = s.id_tema
ORDER BY t.tema, s.nombre, v.mes;
""")

with engine.connect() as conn:
    df_interanual = pd.read_sql(consulta_ventana, conn)

# Mostrar muestra con variaciones interanuales calculadas
df_interanual.dropna(subset=["var_12m_pct"]).head(15)

,tema,serie,mes,valor_mensual,var_12m_pct
12,Actividad,Imacec comercio,2011-01,81.21,13.77
13,Actividad,Imacec comercio,2011-02,75.32,16.81
14,Actividad,Imacec comercio,2011-03,88.14,19.62
15,Actividad,Imacec comercio,2011-04,87.44,9.45
16,Actividad,Imacec comercio,2011-05,88.03,13.50
17,Actividad,Imacec comercio,2011-06,79.92,12.48
18,Actividad,Imacec comercio,2011-07,78.94,9.78
19,Actividad,Imacec comercio,2011-08,78.41,7.78
20,Actividad,Imacec comercio,2011-09,78.05,11.00
21,Actividad,Imacec comercio,2011-10,86.05,7.74
